<a href="https://colab.research.google.com/github/wiz124/chem169-git/blob/main/Li_Harry_RID_019_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Exercise 0
#Exercise 0
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

def count_amino_acids(sequence):
    amino_acids = list('ACDEFGHIKLMNPQRSTVWY')
    sequence=str(sequence)
    return {aa: sequence.count(aa) for aa in amino_acids}

df=pd.read_excel('mtb_with_localization.xlsx')

location=[]
for entry in df['Subcellular location [CC]']:
  if 'membrane' in str(entry).lower():
    location.append('membrane')
  elif 'secreted' in str(entry).lower():
    location.append('secreted')
  elif 'cytoplasm' in str(entry).lower():
    location.append(  'cytoplasm')
  else:
    location.append(None)

df['location']=location
df.dropna(subset=['location'], inplace=True)

y = df[['location']].to_numpy()

amino_acids = list('ACDEFGHIKLMNPQRSTVWY')

aa_counts = pd.DataFrame(
    df['Sequence'].apply(count_amino_acids).tolist(),
    columns=amino_acids,
    index=df.index
)
df = pd.concat([df, aa_counts], axis=1)


x = df[['Length'] + amino_acids].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,      # 30% for testing
    random_state=42,    # for reproducibility
    stratify=y          # maintain class distribution
)

print(X_train.shape)
print(y_train.shape)

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


(873, 21)
(873, 1)


In [6]:
#Exercise 1

loc_to_num=[]
for location in df['location']:
  if location =='membrane':
    loc_to_num.append(1)
  elif location =='secreted':
    loc_to_num.append(2)
  elif location =='cytoplasm':
    loc_to_num.append(0)

df['Encoded location']=loc_to_num
x = df[['Length'] + amino_acids+['Encoded location']].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,      # 30% for testing
    random_state=42,    # for reproducibility
    stratify=y          # maintain class distribution
)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder


logmodel = LogisticRegression(solver='liblinear',max_iter=1000)
logmodel.fit(X_train, y_train)

forestmodel = RandomForestClassifier(n_estimators=100, random_state=42)
forestmodel.fit(X_train,y_train)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

xgbmodel = xgb.XGBClassifier(n_estimators=100,
                             random_state=42,
                             )
xgbmodel.fit(X_train, y_train_enc)
log_pred = logmodel.predict(X_test)
forest_pred = forestmodel.predict(X_test)
xgb_pred = xgbmodel.predict(X_test)

log_accuracy = accuracy_score(y_test, log_pred)
forest_accuracy = accuracy_score(y_test, forest_pred)

xgb_accuracy = accuracy_score(y_test_enc,xgb_pred)

results={
    'Logistic Regression':log_accuracy,
    'Random Forest':forest_accuracy,
    'XGBoost':xgb_accuracy
}
resultdf=pd.DataFrame(results,index=['Accuracy'])
display(resultdf)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:1408: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_label.py:129: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), 

,Logistic Regression,Random Forest,XGBoost
Accuracy,0.981735,0.990868,1.0


Exercise 2

Q1: They should not be in different splits

Q2: They should be in different splits at 50%. At 30%, there may be similarity but evolutionary and functional similarity become harder to ascertain. Thus, 30% is acceptable.

Q3: We can do pairwise sequence alignment to figure out which proteins are similar.


